# Exploratory Data Analysis

## 1. Objective

Analyze the SDSS 17 Stellar Classification, understand the structure, schema, data type and basic statistical properties of the dataset. At the same time, identifying potential data quality issues.

# 2. Dataset Overview

## Context

Stellar Classification, in Astronomy, is the classification of stars based on their spectral characteristics. This dataset aims to classify stars, galaxies, and quasars based on their spectral characteristics.

## Data Dictionary

- *obj_ID:* Object Identifier, the unique value that identifies the object in the image catalog used by the CAS
- *alpha:* Right Ascension angle (at J2000 epoch)
- *delta:* Declination angle (at J2000 epoch)
- *u:* Ultraviolet filter in the photometric system
- *g:* Green filter in the photometric system
- *r:* Red filter in the photometric system
- *i:* Near Infrared filter in the photometric system
- *z:* Infrared filter in the photometric system
- *run_ID:* Run Number used to identify the specific scan
- *rereun_ID:* Rerun Number to specify how the image was processed
- *cam_col:* Camera column to identify the scanline within the run
- *field_ID:* Field number to identify each field
- *spec_obj_ID:* Unique ID used for optical spectroscopic objects (this means that 2 different observations with the same spec_obj_ID must share the output class)
- *class:* object class (galaxy, star or quasar object)
- *redshift:* redshift value based on the increase in wavelength
- *plate:* plate ID, identifies each plate in SDSS
- *MJD:* Modified Julian Date, used to indicate when a given piece of SDSS data was taken
- *fiber_ID:* fiber ID that identifies the fiber that pointed the light at the focal plane in each observation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
import sys
from pyprojroot import here

sys.path.append(str(here()))

from src.exist.config import DATA_PATH
from src.exist.utils import plot_histogram, plot_box

In [ ]:
df = pd.read_csv(DATA_PATH)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### What is important here: 

After a basic overview of the data, the follow conclusions can be made:
- The data is relatively easy to handle with.
- The dataset contains additional metadata and identifier columns that are not relevant to the objectives of this analysis. These variables are therefore excluded from future analysis.
- The columns that will be used in the future analysis are the five photometric bands(**u**, **g**, **r**, **i**, **z**), **redshift** and **class**, as these variables directly represent the measurements relevant to the classification task. 

# 3. Data Quality

In [ ]:
quality = pd.DataFrame(
    {
        "dtype": df.dtypes,
        "unique_values": df.nunique(),
        "null_count": df.isna().sum(),
        "sentinel_-9999_count": (df == -9999).sum(),
    }
)

In [ ]:
quality

### Data quality decisions:

- The sentinel values (`-9999`) were detected in **u**, **g** and **z**. These values are outside the expected range and will need special treatment in the data cleaning stage.
- The type of the photometric value and redshift are correct. Therefore, no changes to the type of those collums.
- The type of **class** collumn will be changed from `str` to `category` during the data cleaning stage. It is a categorical variable, so the `category` type fits better.

# 4. Variable Analysis

In [ ]:
features = ["u", "g", "r", "i", "z", "redshift"]
target = "class"

analysis_df = df[features + [target]].copy()
analysis_df[features] = analysis_df[features].replace(-9999, np.nan)

## 4.1 Target Variable

### 4.1.1 Class Distribution

The first step is to analyze the distribution of the target variable and verify whether the classes are balanced.

The target variable contains three classes (GALAXY, STAR and QSO). The class distribution should be examined to determine if the classes are distributed equally or is an unbalanced distribution between classes.

In [ ]:
analysis_df["class"].value_counts(normalize=True)

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(data=analysis_df, x="class")

plt.title("Class distribution")
plt.xlabel("Class")
plt.ylabel("Count")

plt.show()

#### Conclusions about class:

The dataset is imbalanced, with GALAXY representing the majority of observations, while STAR and QSO have smaller proportions.

## 4.2 Univariate Analysis

### 4.2.1 Photometric Values

Photometric values measure visible light adjusted to match how the human eye perceives brightness.
Remember the dictionary:

- u: Ultraviolet filter in the photometric system
- g: Green filter in the photometric system
- r: Red filter in the photometric system
- i: Near Infrared filter in the photometric system
- z: Infrared filter in the photometric system

In each variable the following questions will be answered:
- What is its range?
- Where are values concentrated?
- Is the distribution symetric? If not, for what side is the tail?
- Are there potential outliers?

#### u - Ultraviolet filter:

In [ ]:
analysis_df["u"].describe()

In [ ]:
analysis_df["u"].skew()

In [ ]:
plot_histogram(
    data=analysis_df, x="u", title="Ultraviolet distribuition", grid=False, kde=True
)

In [ ]:
plot_box(data=analysis_df, x="u", title="Ultraviolet boxplot")

Conclusions:

- The range is from 10.99 to 32.78.
- The values are concentrated in the 20 < `u` < 24 range.
- It appears to be approximately symmetric.
- There is no potential outlier.

#### g - Green filter:

In [ ]:
analysis_df["g"].describe()

In [ ]:
analysis_df["g"].skew()

In [ ]:
plot_histogram(
    data=analysis_df, x="g", title="Green distribuition", grid=False, kde=True
)

In [ ]:
plot_box(data=analysis_df, x="g", title="Green boxplot")

Conclusions:

- The range is from 10.49 to 31.60.
- The values are concentrated in the 18 < `g` < 23 range.
- The distribution shows a slight left skew.
- There is no potential outlier.

#### r - Red filter:

In [ ]:
analysis_df["r"].describe()

In [ ]:
analysis_df["r"].skew()

In [ ]:
plot_histogram(data=analysis_df, x="r", title="Red distribuition", grid=False, kde=True)

In [ ]:
plot_box(data=analysis_df, x="r", title="Red boxplot")

Conclusions:

- The range is from 9.82 to 29.57.
- The values are concentrated in the 17 < `r` < 22 range.
- The distribution shows a slight left skew.
- There is no potential outlier.

#### i - Near infrared filter:

In [ ]:
analysis_df["i"].describe()

In [ ]:
analysis_df["i"].skew()

In [ ]:
plot_histogram(
    data=analysis_df, x="i", title="Near infrared distribuition", grid=False, kde=True
)

In [ ]:
plot_box(data=analysis_df, x="i", title="Near infrared boxplot")

Conclusions:

- The range is from 9.46 to 32.14.
- The values are concentrated in the 17 < `i` < 21 range.
- The distribution shows a slight left skew.
- There is no potential outlier.

#### z - Infrared filter:

In [ ]:
analysis_df["z"].describe()

In [ ]:
analysis_df["z"].skew()

In [ ]:
plot_histogram(
    data=analysis_df, x="z", title="Infrared distribuition", grid=False, kde=True
)

In [ ]:
plot_box(data=analysis_df, x="z", title="Infrared boxplot")

Conclusions:

- The range is from 9.61 to 29.38.
- The values are concentrated in the 16 < `z` < 20 range.
- The distribution shows a slight left skew.
- There is no potential outlier.

### 4.2.2 Redshift

In astronomy redshift is the streching of light waves to longer, redder wavelenghts as they travel through space. Redshift values are used to calculate how far away galaxies, quasar are.

How is redshift distributed?

In [ ]:
analysis_df["redshift"].describe()

In [ ]:
analysis_df["redshift"].skew()

In [ ]:
plot_histogram(
    data=analysis_df,
    x="redshift",
    title="Redshift distribuition",
    grid=False,
    kde=False,
    bins=100,
)

In [ ]:
plot_box(data=analysis_df, x="redshift", title="Redshift boxplot", grid=False)

#### Conclusions about redshift:

- Redshift values are mainly concentrated in the 0 < `redshift` < 0.8 range.
- Median (approximately 0.42) and mean (0.57) showing asymmetry. 
- It has a tail to the right, observed mostly in the `skew()` with the value of 2.5, and in the boxplot graph.
- The graph has many outlier values between approximately 1.7 to 7.